# Midterm Manure Q3 Benchmark

Local VS Code notebook for evaluating ChatbotLP on Midterm Problem 1, Question 3: Manure Management with the dairy-farmer removal incentive.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT

In [ ]:
import os

os.environ["GEMINI_API_KEY"] = ""
os.environ["LLM_PROVIDER"] = "gemini"
os.environ["GEMINI_MODEL"] = "gemini-3-flash-preview"

In [ ]:
import json
import pandas as pd
from IPython.display import Markdown, display

from src.midterm_benchmark import (
    DEFAULT_Q3_BENCHMARK_DIR,
    MIDTERM_Q3_REASONING_PROMPTS,
    MidtermBenchmarkConfig,
    load_benchmark_files,
    run_midterm_manure_q3_benchmark,
    write_midterm_outputs,
)

## Problem Statement And Reference Solution

In [ ]:
files = load_benchmark_files(DEFAULT_Q3_BENCHMARK_DIR)
reference = files["reference_solution"]

display(Markdown(files["problem_statement"]))

reference_summary = pd.DataFrame([
    {"metric": "objective_value", "value": reference["objective_value"]},
    {"metric": "demand_revenue", "value": reference["demand_revenue"]},
    {"metric": "transport_cost", "value": reference["transport_cost"]},
    {"metric": "supply_cost", "value": reference["supply_cost"]},
    {"metric": "supply_contribution", "value": reference["supply_contribution"]},
    {"metric": "total_accepted_supply", "value": sum(reference["accepted_supply"].values())},
    {"metric": "total_accepted_demand", "value": sum(reference["accepted_demands"].values())},
    {"metric": "total_transport_flow", "value": sum(reference["transport_flows"].values())},
])
display(reference_summary)
display(reference)

## Canonical And Paraphrased Prompt Runs

In [ ]:
USE_LLM = bool(os.environ.get("GEMINI_API_KEY"))

config = MidtermBenchmarkConfig(
    prompt_ids=("canonical", "paraphrased"),
    use_llm=USE_LLM,
    use_llm_for_reasoning=USE_LLM,
    fallback_to_reference_fixture=True,
    attempt_solve=True,
    run_reasoning=False,
)
report = run_midterm_manure_q3_benchmark(config=config)

display(pd.DataFrame([report["metadata"]]))
display(report["tables"]["case_summary"])
display(report["tables"]["solve_accuracy"])
display(report["tables"]["interpretation_metadata"])
display(report["tables"]["interpretation_errors"])

## ID-Independent Diagnostic Tables

In [ ]:
diagnostic_table_names = [
    "semantic_count_metrics",
    "parameter_multiset_metrics",
    "topology_metrics",
    "technology_yield_metrics",
    "route_economics_metrics",
    "route_association_metrics",
    "transport_link_attribute_metrics",
    "solver_aggregate_metrics",
    "balance_residual_metrics",
    "formulation_completeness_metrics",
    "solve_correctness_metrics",
    "reasoning_readiness_metrics",
    "active_flow_objective_diagnostics",
    "alias_resolution_diagnostics",
]

for table_name in diagnostic_table_names:
    display(Markdown(f"### {table_name.replace('_', ' ').title()}"))
    display(report["tables"][table_name])

## Q3 Supplier Removal Incentive Diagnostics

In [ ]:
supplier_payment_rows = report["tables"]["parameter_multiset_metrics"].query(
    "metric == 'supplier_bid_prices'"
)
supplier_topology_rows = report["tables"]["topology_metrics"].query(
    "metric == 'negative_supplier_bid_detection'"
)
removal_rows = report["tables"]["removal_incentive_diagnostics"]
route_economics_table = pd.DataFrame([
    {"route": "Eau Claire -> Menomonie", "consumer_bid": -0.5, "supplier_bid": -0.7, "transport_cost": 0.1, "route_net_value": 0.1},
    {"route": "Eau Claire -> Black River Falls", "consumer_bid": 1.5, "supplier_bid": -0.7, "transport_cost": 0.2, "route_net_value": 2.0},
])
q2_q3_comparison_table = pd.DataFrame([
    {"metric": "supplier_bid", "q2": 0.0, "q3": -0.7},
    {"metric": "accepted_supply", "q2": 500, "q3": 1000},
    {"metric": "Menomonie_flow", "q2": 0, "q3": 500},
    {"metric": "objective_value", "q2": 650, "q3": 1050},
])

display(supplier_payment_rows)
display(supplier_topology_rows)
display(removal_rows)
display(route_economics_table)
display(q2_q3_comparison_table)

## Primary Metric Flags

In [ ]:
primary_flags = report["tables"]["case_summary"][[
    "prompt_id",
    "formulation_completeness_pass",
    "solve_correctness_pass",
    "reasoning_ready_pass",
    "route_attribute_binding_pass",
    "route_association_pass",
    "semantic_structure_pass",
    "technology_structure_pass",
    "solver_aggregate_pass",
    "balance_residual_pass",
    "primary_success",
    "failure_type",
]]
display(primary_flags)

## Flow Table

In [ ]:
flow_table = pd.DataFrame([
    {"destination": "Menomonie", "route": "Eau Claire -> Menomonie", "flow_tons": 500},
    {"destination": "Black River Falls", "route": "Eau Claire -> Black River Falls", "flow_tons": 500},
])
display(flow_table)

## Required Reasoning Prompts

In [ ]:
display(pd.DataFrame(MIDTERM_Q3_REASONING_PROMPTS)[["id", "label", "prompt"]])

reasoning_config = MidtermBenchmarkConfig(
    prompt_ids=("canonical",),
    use_llm=USE_LLM,
    use_llm_for_reasoning=USE_LLM,
    fallback_to_reference_fixture=True,
    attempt_solve=True,
    run_reasoning=True,
)
reasoning_report = run_midterm_manure_q3_benchmark(config=reasoning_config)
display(reasoning_report["tables"]["reasoning_prompt_success"])

for row in reasoning_report["cases"][0]["reasoning_results"]:
    display(Markdown(f"### {row['prompt_label']}\n\n{row['response_preview']}"))

## Export

In [ ]:
output_dir = REPO_ROOT / "midterm_outputs"
write_midterm_outputs(report, output_dir)
reasoning_report["tables"]["reasoning_prompt_success"].to_csv(
    output_dir / "manure_q3_reasoning_prompt_success_canonical.csv",
    index=False,
)
flow_table.to_csv(output_dir / "manure_q3_flow_table.csv", index=False)
route_economics_table.to_csv(output_dir / "manure_q3_route_economics.csv", index=False)
q2_q3_comparison_table.to_csv(output_dir / "manure_q2_q3_comparison.csv", index=False)

paper_summary = report["tables"]["case_summary"][[
    "prompt_id",
    "solver_ready_actual",
    "formulation_completeness_pass",
    "solve_correctness_pass",
    "reasoning_ready_pass",
    "route_attribute_binding_pass",
    "route_association_pass",
    "semantic_structure_pass",
    "technology_structure_pass",
    "solver_aggregate_pass",
    "balance_residual_pass",
    "primary_success",
    "failure_type",
    "solve_success",
]]
paper_summary.to_csv(output_dir / "manure_q3_paper_style_summary.csv", index=False)
display(paper_summary)
output_dir